# 05 · Drive → Kaggle Dataset (one-time upload)

Run this notebook **once in Colab (CPU runtime is fine — no GPU needed).**  
It copies everything your training needs from Google Drive into a permanent **Kaggle Dataset**.  
After this, Kaggle notebooks can train completely **without Google Drive**.

### What gets uploaded
| Item | Source on Drive | Size (approx) |
|------|----------------|---------------|
| `asvspoof.tar` | `archives/` | ~8 GB |
| `noise.tar` | `archives/` | ~2 GB |
| `eval_noisy.tar` | `archives/` | ~3 GB |
| `eval_metricgan.tar` | `archives/` | ~3 GB |
| `eval_sepformer.tar` | `archives/` | ~3 GB |
| `train_aug.tar` | `archives/` | ~2 GB |
| `aasist/` (code) | cloned from GitHub | ~50 MB |
| `checkpoints/` | `checkpoints/` | varies |

### Pre-requisites
1. Get your **Kaggle API key**: kaggle.com → Settings → API → *Create New Token* → downloads `kaggle.json`
2. In Colab, click the **🔑 key icon** on the left sidebar → add two secrets:
   - `KAGGLE_USERNAME` → your Kaggle username
   - `KAGGLE_KEY` → the `key` value from inside `kaggle.json`
3. Switch on *Notebook access* for both secrets

> **Runtime:** `Runtime → Change runtime type → CPU` (no GPU needed, saves quota)

## Step 1 · Configure credentials & paths

In [ ]:
# ── Install Kaggle API ──────────────────────────────────────────────────────
!pip install -q kaggle

import os, json, shutil
from pathlib import Path
from google.colab import userdata, drive

# ── Read Kaggle credentials from Colab Secrets ─────────────────────────────
try:
    KAG_USER = userdata.get('KAGGLE_USERNAME')
    KAG_KEY  = userdata.get('KAGGLE_KEY')
except Exception:
    raise RuntimeError(
        "Kaggle secrets not found!\n"
        "Add KAGGLE_USERNAME and KAGGLE_KEY in the 🔑 Secrets panel on the left.")

os.makedirs('/root/.config/kaggle', exist_ok=True)
Path('/root/.config/kaggle/kaggle.json').write_text(
    json.dumps({'username': KAG_USER, 'key': KAG_KEY}))
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print(f'Kaggle credentials configured for: {KAG_USER} ✓')

# ── Paths ──────────────────────────────────────────────────────────────────
# Edit DRIVE_BASE if your Drive folder has a different name
DRIVE_BASE      = '/content/drive/MyDrive/mini_project'
DRIVE_ARCHIVES  = f'{DRIVE_BASE}/archives'
DRIVE_CHECKPTS  = f'{DRIVE_BASE}/checkpoints'
STAGE_DIR       = '/content/kaggle_stage'   # temp staging folder in Colab
DATASET_NAME    = 'mini-project-data'       # will appear as YOUR_USERNAME/mini-project-data on Kaggle

print(f'Drive archives : {DRIVE_ARCHIVES}')
print(f'Staging dir    : {STAGE_DIR}')
print(f'Kaggle dataset : {KAG_USER}/{DATASET_NAME}')

## Step 2 · Mount Google Drive

In [ ]:
drive.mount('/content/drive')

# Verify archives folder exists
if not Path(DRIVE_ARCHIVES).exists():
    raise FileNotFoundError(
        f'Archives folder not found: {DRIVE_ARCHIVES}\n'
        'Make sure you ran notebooks 01–03 first and that Drive is mounted correctly.')

archives_found = sorted(Path(DRIVE_ARCHIVES).glob('*.tar'))
print(f'Found {len(archives_found)} archive(s) on Drive:')
for a in archives_found:
    size_gb = a.stat().st_size / 1e9
    print(f'  {a.name:<30}  {size_gb:.2f} GB')

## Step 3 · Stage all files into a local folder

We copy everything into `/content/kaggle_stage/` first, then upload that folder as a single Kaggle Dataset.
The copy happens on Colab's fast SSD — Drive reads are slow but this runs unattended.

In [ ]:
import shutil
from tqdm.auto import tqdm

# Clean and re-create staging dir
shutil.rmtree(STAGE_DIR, ignore_errors=True)
os.makedirs(STAGE_DIR)

# ── 3a: Copy all .tar archives ─────────────────────────────────────────────
print('=== Copying .tar archives ===')
archives_dir = Path(STAGE_DIR) / 'archives'
archives_dir.mkdir()

for src in tqdm(archives_found, desc='archives'):
    dst = archives_dir / src.name
    print(f'  {src.name}  ({src.stat().st_size/1e9:.2f} GB) ... ', end='', flush=True)
    shutil.copy2(src, dst)
    print('done')

print(f'\nArchives staged to: {archives_dir}')

In [ ]:
# ── 3b: AASIST code ────────────────────────────────────────────────────────
print('=== Staging AASIST code ===')
drive_aasist = Path(DRIVE_BASE) / 'aasist'
stage_aasist = Path(STAGE_DIR) / 'aasist'

if drive_aasist.exists():
    print(f'  Copying from Drive: {drive_aasist}')
    shutil.copytree(drive_aasist, stage_aasist)
    print(f'  Copied {sum(1 for _ in stage_aasist.rglob("*"))} items')
else:
    # Clone directly from GitHub (fresh, no Drive needed)
    print('  Not found on Drive — cloning from GitHub...')
    !git clone --depth=1 -q https://github.com/clovaai/aasist.git {stage_aasist}
    # Apply the numpy compatibility patch
    import subprocess, sys
    repo_dir = '/content/drive/MyDrive/mini_project/repo'
    if Path(repo_dir).exists():
        result = subprocess.run(
            [sys.executable, 'scripts/setup_aasist.py',
             '--aasist-dir', str(stage_aasist)],
            cwd=repo_dir, capture_output=True, text=True)
        print(result.stdout or '  (numpy patch applied)')
    else:
        print('  Repo not found on Drive — skipping numpy patch (will be applied in Kaggle)')

print('AASIST staged ✓')

In [ ]:
# ── 3c: Checkpoints (optional — skip if none exist yet) ───────────────────
print('=== Staging checkpoints ===')
drive_ckpts = Path(DRIVE_CHECKPTS)
stage_ckpts = Path(STAGE_DIR) / 'checkpoints'

if drive_ckpts.exists() and any(drive_ckpts.rglob('*.pt')):
    checkpoint_files = sorted(drive_ckpts.rglob('*.pt'))
    print(f'  Found {len(checkpoint_files)} checkpoint(s)')
    stage_ckpts.mkdir()
    for ckpt in tqdm(checkpoint_files, desc='checkpoints'):
        rel = ckpt.relative_to(drive_ckpts)
        dst = stage_ckpts / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(ckpt, dst)
    print(f'  Checkpoints staged ✓')
else:
    print('  No checkpoints found on Drive — skipping (that is OK for a fresh run)')

# ── Summary ────────────────────────────────────────────────────────────────
print('\n=== Staging complete. Contents of stage dir ===')
for item in sorted(Path(STAGE_DIR).rglob('*')):
    if item.is_file():
        print(f'  {item.relative_to(STAGE_DIR)}  ({item.stat().st_size/1e6:.1f} MB)')

total_gb = sum(f.stat().st_size for f in Path(STAGE_DIR).rglob('*') if f.is_file()) / 1e9
print(f'\nTotal staged size: {total_gb:.2f} GB')

## Step 4 · Upload to Kaggle

This creates a **new Kaggle Dataset** the first time.  
On future runs (e.g. to add new checkpoints), use the "Update existing dataset" cell below instead.

> ⏱ Upload speed depends on your dataset size. A 20 GB total upload takes ~30–60 min on Colab.  
> **Do not close the tab** while uploading.

In [ ]:
# ── Write Kaggle Dataset metadata ──────────────────────────────────────────
meta = {
    "title":    DATASET_NAME,
    "id":       f"{KAG_USER}/{DATASET_NAME}",
    "licenses": [{"name": "other"}]
}
Path(f'{STAGE_DIR}/dataset-metadata.json').write_text(json.dumps(meta, indent=2))
print('Metadata written ✓')
print(f'Dataset will be: kaggle.com/datasets/{KAG_USER}/{DATASET_NAME}')

In [ ]:
# ── CREATE (run this the FIRST time only) ──────────────────────────────────
# If you get "Dataset already exists", run the UPDATE cell below instead.
print('Uploading to Kaggle... (this may take 30–60 minutes for large datasets)')
!kaggle datasets create -p {STAGE_DIR}
print('\nUpload complete!')
print(f'View your dataset at: https://www.kaggle.com/datasets/{KAG_USER}/{DATASET_NAME}')

In [ ]:
# ── UPDATE (run this for subsequent uploads, e.g. new checkpoints) ─────────
# Uncomment and run this instead of the CREATE cell above if the dataset already exists.

# !kaggle datasets version -p {STAGE_DIR} -m "Updated checkpoints / archives"
# print('Dataset updated!')

## Step 5 · Verify the upload

In [ ]:
import time
print('Waiting 30 seconds for Kaggle to process the upload...')
time.sleep(30)

print('\nFiles on Kaggle dataset:')
!kaggle datasets files {KAG_USER}/{DATASET_NAME}

print('\n✅ Done! Your dataset is ready at:')
print(f'   https://www.kaggle.com/datasets/{KAG_USER}/{DATASET_NAME}')
print('\nNext: Open notebook 06_kaggle_train.ipynb in Kaggle to start training.')

## Step 6 · Flush Drive before closing
Always run this before closing Colab so all Drive writes are committed.

In [ ]:
drive.flush_and_unmount()
print('Drive flushed. You can safely close this tab.')